# EDA - Customer Churn (Telecom)

Exploratory analysis of the churn dataset used to justify feature engineering
and model choices in `src/train.py`.

> Note: dataset is synthetically generated (`src/generate_data.py`) with the
> same schema as the well-known IBM Telco Customer Churn dataset. Swap in the
> real CSV at `data/raw/telco_churn.csv` to reproduce this analysis on real data.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df = pd.read_csv("../data/raw/telco_churn.csv")
df.shape

In [ ]:
df.head()

## 1. Target balance

In [ ]:
churn_rate = (df["Churn"] == "Yes").mean()
print(f"Churn rate: {churn_rate:.2%}")
df["Churn"].value_counts(normalize=True).plot(kind="bar", title="Churn distribution")
plt.show()

## 2. Churn rate by contract type

Expectation: month-to-month customers churn far more than 1-2 year contracts.

In [ ]:
(df.groupby("Contract")["Churn"]
   .apply(lambda s: (s == "Yes").mean())
   .sort_values()
   .plot(kind="barh", title="Churn rate by contract type"))
plt.xlabel("Churn rate")
plt.show()

## 3. Tenure vs churn

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
sns.histplot(data=df, x="tenure", hue="Churn", multiple="stack", bins=30, ax=ax)
ax.set_title("Tenure distribution by churn")
plt.show()

## 4. Monthly charges vs churn

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges", ax=ax)
ax.set_title("Monthly charges by churn")
plt.show()

## 5. Key categorical drivers

Internet service type, tech support, online security and payment method are
typically the strongest categorical predictors of churn.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.ravel(), ["InternetService", "TechSupport", "OnlineSecurity", "PaymentMethod"]):
    rates = df.groupby(col)["Churn"].apply(lambda s: (s == "Yes").mean()).sort_values()
    rates.plot(kind="barh", ax=ax, title=f"Churn rate by {col}")
plt.tight_layout()
plt.show()

## 6. Correlation among numeric features

In [ ]:
numeric_cols = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
corr = df[numeric_cols].assign(Churn=(df["Churn"]=="Yes").astype(int)).corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation matrix")
plt.show()

## Takeaways

- **Contract type** and **tenure** are the dominant drivers of churn - short
  tenure + month-to-month contract is the highest-risk segment.
- **Fiber optic internet without tech support / online security** correlates
  with higher churn, likely a proxy for a segment that's price-sensitive and
  under-supported.
- **Electronic check** payment method correlates with higher churn.
- These findings motivate the derived features in `src/data_processing.py`
  (`num_addons`, `avg_monthly_spend`, `tenure_years`) and the choice to try
  tree-based models (which capture interactions like `contract x tenure`
  natively) alongside logistic regression as an interpretable baseline.
